[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/25_Seed_Data.ipynb)

# Seed Data — Practical Guide

The `seed` module bootstraps a knowledge graph from **trusted, pre-known data** (CSV/JSON/database/API sources) before any extraction runs. This gives extraction a foundation to link against instead of starting from an empty graph.

Key pieces:

- **`SeedDataManager`** — registers data sources and builds foundation graphs
- **`create_foundation_graph()`** — turns registered sources into `entities` + `relationships` + `metadata`
- **`validate_quality()`** — checks a foundation graph before you commit it

All examples below were executed against `semantica/seed/seed_manager.py`.

In [1]:
!pip install -q semantica

## 1) Prepare a seed CSV and register the source

`register_source(name, format, location, entity_type=...)` records where trusted data lives. `verified=True` (the default) marks the source as pre-validated.

In [2]:
import csv
import tempfile
from pathlib import Path
from semantica.seed import SeedDataManager

# Write the sample CSV into a session-scoped temp directory so we never
# clobber a companies.csv that might exist in the user's working directory.
seed_csv = Path(tempfile.mkdtemp(prefix="semantica-seed-")) / "companies.csv"
with open(seed_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "name", "type", "industry"])
    writer.writeheader()
    writer.writerow({"id": "c1", "name": "Acme", "type": "Company", "industry": "robotics"})
    writer.writerow({"id": "c2", "name": "Globex", "type": "Company", "industry": "energy"})

manager = SeedDataManager()
manager.register_source("companies", format="csv", location=str(seed_csv), entity_type="Company")


True

## 2) Load records from a registered source

`load_source(name)` reads the source and enriches each record with `entity_type` and `source` provenance keys.

In [3]:
records = manager.load_source("companies")
print(f"loaded {len(records)} records")
records[0]

Status,Action,Module,Submodule,Progress,ETA,Rate,Time,Extracted
✅,Semantica is seeding,🌱 seed,SeedDataManager,100.0%,-,-,0.00s,-


🔄 Semantica is seeding: Loading seed data from CSV: /var/folders/7s/bvvstgs10y963tz6_4bbnklr0000gn/T/semantica-seed-eu9__ep1/companies.csv 🌱 seed SeedDataManager |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

loaded 2 records


{'id': 'c1',
 'name': 'Acme',
 'type': 'Company',
 'industry': 'robotics',
 'entity_type': 'Company',
 'source': 'companies'}

## 3) Build the foundation graph

`create_foundation_graph()` converts every registered source into graph-ready entities and relationships. Entities carry `confidence: 1.0` — seed data is trusted by definition.

In [4]:
foundation = manager.create_foundation_graph()
sorted(foundation.keys())

['entities', 'metadata', 'relationships']

In [5]:
foundation["entities"][0]

{'id': 'c1',
 'text': 'Acme',
 'type': 'Company',
 'confidence': 1.0,
 'metadata': {'industry': 'robotics', 'source': 'companies'}}

## 4) Validate quality before committing

`validate_quality(foundation_graph)` returns `valid`, `errors`, `warnings`, and `metrics` so you can gate bad seed data before it pollutes the graph.

In [6]:
quality = manager.validate_quality(foundation)
quality["valid"]

True

## Summary

| Task | API |
|---|---|
| Register a trusted source | `register_source(name, format, location, entity_type=...)` |
| Load records | `load_source(name)` — adds `entity_type` / `source` keys |
| Direct file load | `load_from_csv(path)` / `load_from_json(path)` |
| Build the graph | `create_foundation_graph()` → `entities` / `relationships` / `metadata` |
| Gate bad data | `validate_quality(graph)` → `valid` / `errors` / `warnings` / `metrics` |

See also `semantica/seed/seed_usage.md` for `load_from_database`, `load_from_api`, and `integrate_with_extracted`.